# Práctica 1 · 04 — Contraste estación–reanálisis

**Cambio Climático · Ingeniería Ambiental · ESPAM MFL** · Material CC-IAM-P1-04 · Versión 3.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/P1_04_estacion_vs_reanalisis.ipynb)

**Objetivo.** Comparar las mediciones de una estación con lo que ERA5 y CORe dicen para ese punto, en todas las variables que mida la estación.

**Producto.** Tabla de métricas, ciclo anual comparado y figuras (paso 2 de la práctica).

**Requisito.** Haber ejecutado `P1_01_descarga_datos` y tener los datos de la estación en un CSV con el formato de `plantilla_estacion.csv`. Solo `fecha` es obligatoria; se comparan las columnas que tenga.

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Escriba solo en la celda **1** y ejecute las demás en orden.

<details><summary>Secuencia de la práctica y variables</summary>


| Orden | Cuaderno | Qué hace | Deja listo |
|---|---|---|---|
| 1 | `P1_01_descarga_datos` | Descarga ERA5 y CORe solo para el recuadro de su zona | `CC_datos/<zona>/` |
| 2 | `P1_02_series` | Recorta con el límite y promedia sobre la zona | Series diaria y mensual |
| 3 | `P1_03_climatologia` | Climatología 1995–2014 de las 8 variables con los dos reanálisis | Tablas y figuras (paso 1) |
| 4 | `P1_04_estacion_vs_reanalisis` | Contrasta una estación con ERA5 y CORe | Métricas y figuras (paso 2) |


| Variable | Columna | Unidad |
|---|---|---|
| Temperatura media, máxima y mínima a 2 m | `tmed`, `tmax`, `tmin` | °C |
| Precipitación | `pr` | mm/día · mm/mes |
| Humedad relativa a 2 m | `hr` | % |
| Humedad específica a 2 m | `q` | g/kg |
| Presión atmosférica en superficie | `ps` | hPa |
| Viento a 10 m: componentes, velocidad y dirección de procedencia | `u10`, `v10`, `ws`, `wd` | m/s · ° |

</details>

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del estudiante y de la estación

In [ ]:
ESTUDIANTE = "Nombre Apellido"
PROVINCIA  = "Manabí"   # la misma en todos los cuadernos
CANTON     = None       # None = toda la provincia; o el nombre de un cantón, p. ej. "Portoviejo"

ESTACION   = "Nombre de la estación"
CODIGO     = "M0000"            # código INAMHI, si lo tiene
LAT, LON   = -0.83, -80.18      # grados decimales; sur y oeste negativos (ejemplo: reemplazar)
ALTITUD_M  = 15
ARCHIVO_ESTACION = "estacion.csv"   # súbalo con el ícono de carpeta o indique su ruta en Drive

# Criterios de completitud de un mes con datos diarios (ajustables; se declaran en el informe)
MAX_FALTANTES, MAX_CONSECUTIVOS, MAX_FALTANTES_P = 4, 2, 0
GRADIENTE_T = 0.0065   # °C/m, para llevar la temperatura del reanálisis a la altura de la estación

## 2. Entorno de trabajo

In [ ]:
import os, sys, pathlib, subprocess, datetime, platform, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, variables as V
from herramientas.reanalisis import era5, core, release
from herramientas.estaciones import calidad

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")

import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, geopandas as gpd
Z = limites.zona(PROVINCIA, CANTON)
GEOM, S = Z["geom"], Z["slug"]
DATOS = BASE / "CC_datos" / S            # datos descargados (cuaderno 1)
RES = BASE / "CC_Practica1" / S          # productos de la práctica (cuadernos 2 a 4)
for c in (DATOS, RES): c.mkdir(parents=True, exist_ok=True)
print(f"Zona: {Z['nombre']} ({Z['tipo']}, DPA {Z['codigo']}, región {Z['region']})")
print(f"Datos en {DATOS} · resultados en {RES}")

## 3. Datos de la estación: lectura y control de calidad

Fechas repetidas, valores físicamente imposibles (se anulan y se cuentan), dirección del viento con calma (se anula) y completitud de cada mes. Un dato dudoso no se corrige a mano: se marca como faltante y se reporta.

In [ ]:
est, info = calidad.leer(ARCHIVO_ESTACION)
DIARIO = info["frecuencia"] == "diaria"
est, anulados = calidad.limites_fisicos(est, DIARIO)
EST_M = calidad.mensual(est, DIARIO, MAX_FALTANTES, MAX_CONSECUTIVOS, MAX_FALTANTES_P)
print("Frecuencia:", info["frecuencia"], "· variables:", info["variables"], "· duplicadas:", info["duplicadas"], "· anulados:", anulados)
print("Periodo:", EST_M.index.min().date(), "a", EST_M.index.max().date())
print("Meses válidos:", EST_M.notnull().sum()[lambda s: s > 0].to_dict())

## 4. Los reanálisis en el punto de la estación

- **ERA5:** la celda más cercana. La temperatura se lleva a la altura de la estación con 6,5 °C/km y la presión con la ecuación hipsométrica, usando la altura del terreno del modelo.
- **CORe:** interpolación bilineal al punto, sin corrección de altura (no se dispone de su terreno). Su presión no es comparable directamente con la de la estación si las alturas difieren.
- **Viento:** el reanálisis da el viento a 10 m; muchas estaciones lo miden a menor altura, lo que reduce la velocidad medida.

In [ ]:
inv = json.loads((DATOS / "inventario.json").read_text(encoding="utf-8"))
A = {k: (None if v is None else DATOS / v) for k, v in inv["archivos"].items()}
n, o, s, e = inv["recuadro_era5"]
assert s <= LAT <= n and o <= LON <= e, "La estación está fuera del recuadro descargado: revise coordenadas o zona."
P = dict(lat=LAT, lon=LON)

ALT_ERA5 = float(era5.terreno(A["era5_terreno"]).sel(P, method="nearest"))
DZ = ALTITUD_M - ALT_ERA5
CORR_T = -GRADIENTE_T * DZ

def corregir_era5(df):
    df = df.copy()
    for c in ("tmed", "tmax", "tmin"):
        if c in df: df[c] = df[c] + CORR_T
    if "ps" in df: df["ps"] = df["ps"] * np.exp(-9.80665 * DZ / (287.05 * (df["tmed"] + 273.15)))
    return df

def punto(ds, fuente, vecino=True):
    p = ds.sel(P, method="nearest") if vecino else ds.interp(P)
    df = p[[v for v in ("tmed", "tmax", "tmin", "pr", "hr", "q", "ps", "u10", "v10") if v in p]].to_dataframe()
    df = df[[c for c in df.columns if c not in ("lat", "lon")]]
    if {"u10", "v10"} <= set(df): df["ws"], df["wd"] = V.viento(df["u10"], df["v10"])
    return df

em = punto(era5.mensual_estandar(A["era5_mensual"]), "ERA5")
if A.get("era5_diario"):
    d = xr.open_dataset(A["era5_diario"])[["tmax", "tmin"]].sel(P, method="nearest").resample(time="MS").mean().to_dataframe()
    em = em.join(d[["tmax", "tmin"]])
REA = corregir_era5(em).add_prefix("ERA5_")
if A.get("core_mensual"):
    REA = REA.join(punto(xr.open_dataset(A["core_mensual"]), "CORe", vecino=False).add_prefix("CORe_"))
REA.index = REA.index.to_period("M").to_timestamp()
PARES = EST_M.join(REA, how="inner")
FUENTES = [f for f in ("ERA5", "CORe") if f"{f}_tmed" in PARES]
print(f"Terreno ERA5 {ALT_ERA5:.0f} m · estación {ALTITUD_M} m · corrección T {CORR_T:+.2f} °C · {len(PARES)} meses en común")

## 5. Métricas mensuales

Solo pares con dato en ambas fuentes. **Sesgo** = media(reanálisis − estación). **RMSE**: error típico. **r**: si sigue las subidas y bajadas. Para la dirección del viento, las diferencias se toman en el círculo (−180° a 180°) y no se calcula r.

In [ ]:
COMPARA = {"tmed": "tmed", "tmax": "tmax", "tmin": "tmin", "precip": "pr", "hr": "hr", "q": "q", "pres": "ps", "vv": "ws", "dv": "wd"}
filas = {}
for obs, rea in COMPARA.items():
    if PARES[obs].notnull().sum() == 0: continue
    for f in FUENTES:
        if f"{f}_{rea}" in PARES:
            filas[f"{obs} · {f}"] = calidad.metricas(PARES[obs], PARES[f"{f}_{rea}"], relativa=obs == "precip", circular=obs == "dv")
METRICAS = pd.DataFrame(filas).T
display(METRICAS)

## 6. Ciclo anual de cada variable medida

In [ ]:
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
col = {"ERA5": "#C2531F", "CORe": "#4C78A8"}
medidas = [o for o in COMPARA if o != "dv" and PARES[o].notnull().sum() >= 12]
CICLO = {}
fig, ax = plt.subplots(1, len(medidas), figsize=(4 * len(medidas), 3.8), squeeze=False)
for a, obs in zip(ax[0], medidas):
    cols = [obs] + [f"{f}_{COMPARA[obs]}" for f in FUENTES if f"{f}_{COMPARA[obs]}" in PARES]
    g = PARES[cols].dropna().groupby(lambda t: t.month).mean().reindex(range(1, 13))
    CICLO[obs] = g
    a.plot(MESES, g[obs], "o-", color="#12263A", label="Estación")
    for f in FUENTES:
        if f"{f}_{COMPARA[obs]}" in g: a.plot(MESES, g[f"{f}_{COMPARA[obs]}"], "s--", color=col[f], label=f)
    a.set_title(obs); a.tick_params(axis="x", labelsize=7, rotation=90); a.legend(fontsize=7)
fig.suptitle(f"{ESTACION} · ciclo anual (meses con dato en todas las fuentes)"); fig.tight_layout()
fig.savefig(RES / "P1_04_ciclo_anual.png", dpi=300); plt.show()

## 7. Comparación diaria (si la estación tiene datos diarios)

Con datos diarios se ve si el reanálisis capta los días de lluvia, que es lo que importa para los índices de extremos. Cada fuente define el día de forma distinta: compare con cautela.

In [ ]:
if DIARIO and A.get("era5_diario"):
    D = est.join(corregir_era5(punto(xr.open_dataset(A["era5_diario"]), "ERA5")).add_prefix("ERA5_"), how="inner")
    if A.get("core_diario"):
        D = D.join(punto(xr.open_dataset(A["core_diario"]), "CORe", vecino=False).add_prefix("CORe_"), how="inner")
    filas = {}
    for obs, rea in COMPARA.items():
        if D[obs].notnull().sum() == 0: continue
        for f in FUENTES:
            if f"{f}_{rea}" in D:
                filas[f"{obs} · {f}"] = calidad.metricas(D[obs], D[f"{f}_{rea}"], circular=obs == "dv")
        if obs == "precip":
            for f in FUENTES:
                ll = D[["precip", f"{f}_pr"]].dropna() >= 1
                filas[f"precip · {f}"]["acierto_dia_lluvia_%"] = round(100 * (ll["precip"] == ll[f"{f}_pr"]).mean(), 1)
    METRICAS_D = pd.DataFrame(filas).T; display(METRICAS_D)
else:
    METRICAS_D = None; print("Sin comparación diaria (estación mensual o sin ERA5 diario).")

## 8. Guardar

In [ ]:
METRICAS.to_csv(RES / "P1_04_metricas_mensuales.csv", encoding="utf-8")
pd.concat(CICLO, axis=1).to_csv(RES / "P1_04_ciclo_anual.csv", encoding="utf-8")
PARES.to_csv(RES / "P1_04_pares_mensuales.csv", encoding="utf-8")
if METRICAS_D is not None: METRICAS_D.to_csv(RES / "P1_04_metricas_diarias.csv", encoding="utf-8")
meta = {"material": "CC-IAM-P1-04 v3.0", "estudiante": ESTUDIANTE, "zona": Z["nombre"],
        "estacion": {"nombre": ESTACION, "codigo": CODIGO, "lat": LAT, "lon": LON, "altitud_m": ALTITUD_M, **info, "anulados": anulados},
        "completitud": {"max_faltantes": MAX_FALTANTES, "max_consecutivos": MAX_CONSECUTIVOS, "max_faltantes_precip": MAX_FALTANTES_P},
        "era5": {"altitud_terreno_m": round(ALT_ERA5, 1), "correccion_T_C": round(CORR_T, 3), "presion": "ecuación hipsométrica"},
        "core": "interpolación bilineal al punto, sin corrección de altura" if "CORe" in FUENTES else None,
        "meses_en_comun": len(PARES)}
(RES / "P1_04_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(sorted(p.name for p in RES.glob("P1_04_*")))

## 9. Preguntas para el capítulo 1

1. ¿Qué variable reproduce mejor cada reanálisis y cuál peor? Use sesgo, RMSE y r.
2. ¿Cuánto del sesgo de temperatura y de presión explica la diferencia de altura entre la estación y el terreno del modelo?
3. ¿El sesgo de la precipitación y de la humedad cambia entre la estación lluviosa y la seca? ¿Qué humedad reproduce mejor cada reanálisis, la específica o la relativa? ¿Por qué un error de temperatura afecta a la relativa y no a la específica?
4. ¿La dirección predominante del viento coincide? ¿Por qué la velocidad medida suele ser menor que la del reanálisis?
5. Si hizo la comparación diaria: ¿qué porcentaje de días de lluvia acierta cada reanálisis y qué implica para los índices de extremos?

**Para el informe:** periodo comparado, meses válidos, criterios de completitud, correcciones por altura y definición del día de cada fuente.